# Laboratorio 1.3 · Ingesta desde bases de datos

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la base transaccional de la tienda en línea

Vas a sacar los pedidos de una base **PostgreSQL** (que corre dentro de este Colab) hacia un warehouse **DuckDB**
con tres métodos, y vas a comprobar con datos cuál dice la verdad:

| Método | Cómo detecta cambios | Lo que vas a descubrir |
|---|---|---|
| Carga completa | No detecta: lee todo | Exacta, pero lee la tabla entera cada vez |
| Incremental (watermark) | `actualizado_en > última marca` | Barata, pero pierde borrados y transacciones que confirman tarde |
| CDC (log de cambios) | Lee el WAL de PostgreSQL | Ve todo, incluidos los borrados |

> La primera celda de preparación instala PostgreSQL en la máquina de Colab y tarda alrededor de un minuto.

## Preparación

In [ ]:
%%capture
!pip -q install "psycopg[binary]" duckdb sqlalchemy python-dotenv

In [ ]:
# PostgreSQL dentro de Colab, con wal_level = logical para poder hacer CDC
!apt-get -qq update > /dev/null && apt-get -qq install -y postgresql > /dev/null
!service postgresql start > /dev/null
!su postgres -c "psql -qc \"ALTER SYSTEM SET wal_level = logical;\"" && service postgresql restart > /dev/null
!su postgres -c "psql -qc \"SELECT pg_drop_replication_slot(slot_name) FROM pg_replication_slots;\"" > /dev/null
!su postgres -c "psql -qc 'DROP DATABASE IF EXISTS tienda;' -c 'DROP ROLE IF EXISTS tienda;'"
!su postgres -c "psql -qc \"CREATE ROLE tienda LOGIN SUPERUSER PASSWORD 'tienda-colab';\" -c 'CREATE DATABASE tienda OWNER tienda;'"
!su postgres -c "psql -tAc 'SHOW wal_level;'"

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
zipfile.ZipFile(io.BytesIO(base64.b64decode("UEsDBBQAAAAIAESUQV2bVl1EXgAAAGUAAAATAAAAaW5nZXN0YS9fX2luaXRfXy5weRWKMQqAMBAEe1+xXKWND7AURUQLQT8Q4qEHkkjuLPy9sZphGCKaxLAzJBys5rJe8E5jDsZHcntMDfyTclnk5ksC6/93zqLiRb/N8DFgee3MKNt1wBjUxB7jqiai4gNQSwMEFAAAAAgARJRBXceuKJ5TAwAA+gYAAA8AAABpbmdlc3RhL2xvZ3MucHmFVc1u3DYQvu9TEOyFChSiDnooFtAhQG0kResU+enFMBY0OVozoUiVP8Y6hh8ppzyCX6xDipJ206DZg5bUDL+Z+eYbilL6h9vvtd0TCNEnGZMXym2JFEoQuAMbHVpIsoKYpy8WBPn93ZtL8k8CEoCMCRQQ6WxIJgqPq8ERJaILnFK60cPofCQfg7Pz2kzR5m24D/MyJa02vXdDBoCoByDVMu9bkp+fnYXJbxTx1uib2e0v3G42bz9c7l7/RrqCx/PjF9bwWzhcbc9eXBNCfiJaYVm611KUuuAjyCT101eb8x8NREEUGDLqEYzGYJuNNCIEcuH8gLVlAlitg0/vIvhmu0FwPNiTvrxjAUzfEg97jcy6as+/DCoww4flTf7RGOh2qZXnCvMiRDGMbEbh0gN6KKTiczezwVOUDdfB1cDl2Aiyo4M2RgfABqlAm/Y0ntV3YDDkgm2w38YKJPrUcZLBsece4p8QgtgD+xbVJ7vTCp2nRqzWx9P6eRpzrQyhkD+/VNgSKrFiF2hLHh6bZjml+zU+HORO295tT2IX3CsK3jtPr5Hg3AE+kXJ+kDBG7Sz7D0jDw2h0zKcDa66en10vqB5wHmwRMFdpGAMrMVoCqHgPOxGk1t2FMAHFia0XOAYdgjeomawEJL7Xexwpv0PBBCa8vNV3rrvErrWkNKCjry8v3tAqD5yac3v39AWnzYU6f4HgTpQpc/h/3xI3SqxEmAHNiJNlTCo04TlZU8YvA0Z/v5KE08ZDVC5F7mHODRhY6RSKuaMp9s9/pRPnUBgj7CV2R9+kCOeZ15b8LUya1keSHnE+ygbLROLn6cDm5usFPKO4zVqu4OjAAxqz4ljhYX1/K6wy4AOXBoRnq2H0bhR7FA1GKJwXy8zLGvVdxCEZXk0wbC26OfbP4ZfhZcej3awRhVIzTD022VCLle+Vgnz9zA1u+Cg8NocPn5T2bNqE7r1P2C44oP527lPZrvpWOkh3VMWFNjAHr7BZd99r1XL8h0V9p7BycLJWuaND1e8kQGbdcOMXwc4JZt225NmzaVxXAb+dBqyosn5BkDwyuZ18ZsJ2jkBrhbsqE+Vw/HuN12539uLnZpHz/wmLo3HSEmZaM4YDJtI9zFfKtmbx2Gz+BVBLAwQUAAAACABElEFdsSG/0fMJAABGGgAAEwAAAGluZ2VzdGEvYXJjaGl2b3MucHmlWN1uHLcVvt+nICYIMJtuJrLTGIaaCeD6pxCayIYUpxeKMODOcLWMZ8gxyZElK+q7+LLXvmsveqEH6iv0O+Rwflaya6MLw+KQh4eH53znj0mSHKgzYR1nlWDclFt5ru0+PpwoS3nzXq1YqZUz3GnLWm1Ya/S5EJU2K1ZzVUl1xt5qJUDWcSOUE4qzS9ZwJTcSfHWWJMlCNtjqWGnP43DL7baW6/j5m9Uqjmt9dga28VPbOLLbzsl6sTG6YRV3wslGsH4tfq8Y/U8SBbqWOzonkr3A5yLyayE/twz/2ipQl1turHCF0qbhtXwrTNxIy0Xrd3vKDFLauHj08rA4eLJi4hwK0IvF4+c/vvzp8NFxcXD4y9PDnx8dHTxnOTtJ7KsuWbGkErY0si2lVvRZcuVkxSsat0ZguuiUdNxITVMN7lJxGm0E5CtKnCiS08ViUdbcWvbUGG0eQ1qwSJ9elKJ1YLzcXzD8oPwnouFWcrrpRmIHM8TmLU3sM1FHozOlmRWs7da1LDkTiq2NVm+Ft99iUYkN9M/vf/cgNZ3jPfstrtVbMutXl37hjXRbpluhPDVkN+tkSZrehI302wBN61q/7mBExaQTJq15s674PttkRuA299j337N7D5Yrtk6S5bjTH511LRk9DSzCsUa4ziisbcVFJQnXkCcI73W+lqYAkhtZ6hRWcFLp1bAyquypnxGkhE4xJ8jKvAasa40jlG7WBkIqXknBzgVTnYLGQBlVyVkjKsmt1x3x7M+CtgiA8ejldC1rvftkzatKmjR82Pxn0wHS4kJaV+hX/jPsGoTKBwak80LxRkT+GX2wP7Akc02bhH3xsmlkEKa1hcbbmpdiWFixQcygQR8TAMvepE2HZcPz+3v4jbo77s4EZC8lZ7yDqm/eOcITokupK8SEkvuwAmVWopYNcI5YkrFjaaBJXJuzJ0+PH7/889HB0Yow6aeev3h69OhoUGcjfgN28tEpAyazdTA4kYyHaQVKvyODWNpHLLnpeYgamE86t/n6YfLJwHXiwpEte5T2ilhmlQB7kU6Phu3IPW2e9OrtzeDM5chvoghwRZDMjhE9N/CHZWZplPoTV5FQEL/Vn37/1SV0aD8XzOv93/PwceFDhxxSgJx4zFXSYzfZDxANCiUAUYSa3AgE8wsmE87J/vSc1cxhkzWMBU0/BJG/T4asY5wljafJf/7172TpAyDCt+FFLZXgA2UNBcu2p8psW0tHBDZdnuydnuw/3Du97kFaC2EQIlURs1Zvx/g5AvVHIchnHx//wnhLMU9VmuJhpKSbjNkuY0/EeSdqgDR9wh1/Zsi3sAlYRjRZ3YqtywGt4xwlAQTuYI6N31Igz/C05PA5O4lw456Mt0BjlUaxkEVEm5xmv2mp4r7lZJ83J5kXX1+EQU5+JFVZd9IgaftzV6wVRrNLzhA8q5v32HkmCcYV7wXcYF9beYgXwFOvx+hC+SjPDB2nK2SRNt+Rdo6E4QeHkbj/QJ2dCZcm/SxlvIww4ba6s7CO3aFrZC0sCO7m/QXZ8vivL5mVCGiGwk+jA5z2WbK3d+/+twnpRVgiVDf/bEgfmP2AqO6yFfmVNsgqCgwIkOos8UksTK4IDQg2yGVT3dRdo7iFweCjDQBLsScQ5jnzFcH1B24gFMogkSftJTRAlQJQvYYxPPLzATzB+BteO9QRgjCGCia9S4Ql+9qvVZssTNqwFxIN2ycQ5BKxcVZdbJKrneBwvd9vZfEY7zaDD11ZqlawM/JfXvcR0OOrohDqU1YvUH6n3Cd3FFSnPZdQVZ16Zv0445aslUYjIWI4k4UQshy2TauwuH0292ls+vIscoifH93sxx382kwYDWXgaXA8pwvVAZSyTOfLY04ptTCUUobDDpR78Mdk5LlbTt7N+hbV7RMGjrMadOAWq+/0FsmKPKThbmLYMKELT3fndSqXEcdF8ONnPrJSmQg81uISfqpC9AJDlGjnvNYGwJfq/OYdgIpGxaGOlDf/oDVUZZPWJOBdnXtA295mo2ozaRWHj/5+t/bmy7N7zpcCDMNUcE+EiSIkCQoQoBmE8JEBGniD4PCxBJDcvifiEMo76p2yqmtamxL/CAZgbOltI0sURRS5bWdEwW0pZf6Mo/BZzopmyPT3iVA4CY2QVJW4SCuj21B7riaCDR3IT0OzNyTXo5BNKIkyp5FYUTn3FtySBnzLWemMvRCGqhe/WA8roMO4okrUkeFCExJSxoYVEEu6okitqDdDGzpRHs1nFKRiwR1J2DcsKSa9KamunnAWymdA6zlPGCJCKu1GvpmvyGcGm2jy5HQ3H594G9WaVzatlx4NNeFgZOjTLGWndMiwfVk6r3dIlDrGkUktccmLQau9XqgxK/qyboYsLxJXl6kIiZQIUeTm+WwLKCrWUwS+gSbRr0LaE8MNBrUBUYNEsaAwvThffdWT3WWoz+x+6DdW6gMXVAzUKN/S4E797h0ye2OAu3TiPL14d3kKld4b3tUuJ6+ilupXlcS+iJP7yrfcRM31pdKmI+iOpeeAVCyF0MHNGTVS/KIIXqXzvWzvu7FENfwNQ2Zs0UuJC47eCxn8B4TAEhbkxHdMtJifPMDgK3TvLH3BDXzL+Z07rzIeDV7U+JKTT4r/1dx1QuM1MADp6PZzInoUmD4VxBpjcvoMrdsJIMILShq7kQKtOkKzLqYbkqjaPGq4p849kkLPQkjOtyf79+6f3lkdB5wCaOhOQ4u4SSZWya8mH9ezJr4gq+SDyhBRMJHQ36FZQFmK7yAdBuMp37BBxFk3vvsoQWeQVeg1hKF53+/fvlC3tZcI8qKHGJaWS58kHa8JADWra3F28z7W8Kt59/E/u6PQnGrnHxZq+Fa1IbjTaNLaBCpueU80WcIVw3YYPAx8iw1cRxhMnp8mzWm49uu5YosR0p+sYNSowezXmbtwyYisD6n69VzRHvSk1xAgQkwmdw8d1+SqdwSa5S0g+7awEGp6lVV84sz+9ujo8ODwL58CaM8o39H2ylshN7pDgUDDFft2OTict9AP0/gyanzijGOgDk7zKdIET6RXjSCLd8pBQoDmAz3N6InxDrevZDpVyCqPj6rIA7G8zJR+k8b33axz5bTzvbNdGURGo3JF+tjP7n15TVXJbr/ObIeaklMz2Ny8u5CNr12upsrL9r6k7mWohYugIF8Hh+G4FJ9TaG3u8355mmg9zXZc8zEHMU/0fckHL48CU4di2s/bVpQ5+m04c2UnJXsRFOqZBaXOwlmfKGaOF+a8z6EkxM18/XvL03qcXIW/18nc/dqQeZKPR7pw1NwH0RaicO33kzOumC9Eh7J1Jxn9XxBGOfPp2P0YbteX6G/DCWDtfJtXWPlWDC+pVD3sXPyjh322I+ymz8/LmXMlfOSuO33D4r9QSwMEFAAAAAgAxJRBXenfvIhiAAAAagAAABYAAABpbmdlc3RhL2JkL19faW5pdF9fLnB5BcFBCsJADAXQvaf4ZFVBCuLObd14jG8aytCZTEkGxHN5BC/meyLy9M1yEKvlaqjEi2kYQU+qlu6sdyhjI7S3o9rgBcU1rJkPVhw98OawaIwdHyyPBVNalt/XcZ1v51lETn9QSwMEFAAAAAgAS5RBXYETOm1QAgAAOgUAAB0AAABpbmdlc3RhL2JkL3NxbC9wb3N0Z3Jlc3FsLnNxbK1TPW/bMBDd/Stu6GADdoBmjNFBls4uUZlyKQqJuxiMxDosJNGhZDTof+rUn5A/1tOHlcSDM7QcuNy7u/fe3c1msFCVhtqpslJpamypcsg05Apqo8tMgS4hf/5dagXjja3qvdPx13Ay8gV6EkF6ixCBLYFHEvCOxTKGNKfMWlcwHgE9k8Hrt2ArxiWskKOgCgEsthDg0ktCCV4MLEAumdzCRrC1J7bwBbfTtk5pi3unhzoS72TblSdh2CFSc8xUdgmh0vqocvNLZXZHwiRbYyy99UZ+G4ADmdL+HE9Gk/noktaDzkxm/7fU3sFdX6+vMzAUuESB3McXr8cmm3S5uqpJ3SUTalvTkIfHkzUK5o8/Xk/henKGLWypaQkume5072aHeNfSf5nEbAaxKRuNGmgpM5PqKaSqWdMn2mFa4Oc/JZiSSBXkC8l0OrXOdQut7ulPbXHIda1OU2U8wLuzqZrsaddPdndGNOIvM38bmlJazzFUoA65SVXHp7RQGV0ciAbROWpHfIdc11ArlEtVc3d1g7M38HgkyhYe1F418Xs60qumcvSDgq0fZEITebCuTWSc+Sw6ne7pnKn7tGmft5jqSOrL78YVHbGrkweRoJ3ahJ6PsEy4LxmprG2q3Jl66ipQJoLH1MLs99o1e/zhw2iBtKHtXDneXp1ZdvOpYzxvAV2BBjcfIQ8oG0KPrxJvhXDID/vqMScTAxFtQAq2WqFohvP23HYtuVezmA832qe8BS5wGQmEZBO0aoe0lhCFAD3/M4jolvqgn8h3bZiP/gJQSwMEFAAAAAgAxJRBXSV/EMSrAwAAbQcAABYAAABpbmdlc3RhL2JkL2NvbmV4aW9uLnB5bVXBcts2EL3zK3bQzISMZcbHjjs6pLE70xm3daSkPbgeBiKXEloQYADQsarRJ/XQySf4x7pLkLRlmQcRxGJ33759Cwkh3luD9+rhmwEJWsJKeoTgpPGyLJU1UkN6bX1YO1x+uAILv2zpncEWSoey7B0r1ID+S4eNzIUQiWpa6wJYn9TONtDKsNFqBcP2NX0m0eK/aKnLDTbb0chBAxZo1srg80N53B7PflpcJQmDmvcx06KolcaiyPJWOjQB3oIgZ5FcLH7+/XKxpHM70cZaeP8cnnydtH5b2nYtZiCa7WDuFyftNm7skySpsO4xuqKxwbq0/53BhuLMoO3QBTuDznfSKVowmbQtvf9qXTX/laiewR3Z5IqATvvix4vi+t1y+cdviwuRnSdAD/F4JaG0hnrh8eE/CaYzpSRvQMONIlOt1p2LPTgHaptGpF5AZ+SUhL+JCuuM7VvDoce8xMe0tI7aRfzeKWdNvsaQHsHMemdVPzopD1xSBMyPk4pgLDoTVIOXzhFBtfhJ6tBL6wVMsDtKs4fU4Z3yEkIHjCgTMXPnNCGmpudRJOnQ1pu+B7dMOzojG5xP/E8Mj4vYqfnQLhLRfOxZJYPkds35JybEe+K+Fw1xbbAMhXRrT7rYCWVUKErbNNJULJTl5Ufgmot/iA9yeX1ydnZ+dvZa7PfMWI8Q5vNRWoCaeNrt+zQOQ+fMofJTKpYBWl20jrhRZj3/6DpSz5s3PazsQIvD8KXR+VFB78kKVJeWfgYP/5pKlUiTG5xar9HlcEmza8u/0ZGaaKYIE/pgQVvYSDpJY80CRIMVwaebQDq6GErpJikFAmOp3pTH8C3UYhch5JWSmhjLuR/7nGvOcgJTFexBOEtbcU2iC/Xp90ODv6qwgcF/hfRKM5Aeykd9EZUvxO+JfTrX03l+yhzvsSwqp+7QFWRPe9AZmb6DYeiBqm3DoFBPo8RFl7xMt8zCq1esWZJwTTPI45ZNKbiThwm9MiwNisEi9MSO+NOI/C9LBWmoiUkNykTqct9qFTRV5KlYKs/YADr31KE2zegtXfDMSypOT0WWHSTiUBNUDpn60TPaeO8ZmpgwFT9QsEPYA79TwGPry2xODqMke7EXFfoKi3hJpWW9fhTlO9dIJjUOhaQhdUG5gWCPZfxP+byqPvNejJBvZaMn1a346qKYN2JVidvnQzRezavqRvQrQXcDf/DUD+snttsbEe+A8dhwe4yffB/Q+kU26Dm+zdnpaFfcZsn/UEsDBBQAAAAIAPCUQV0th+2/LggAAPAVAAAXAAAAaW5nZXN0YS9iZC9zaW11bGFkb3IucHmtWM2O2zgSvvspCA0WkXfcgjuT7KSN9QKOW5N4Ybt7/TMTIGgItETbHEikhpKSdBr9CPMQc8whhyC3ufrFtoqkbMk/3dnBGkiapMiqYvGrr4p0HGfKkyKmkVQkYiSmJOdMRJQwQeLNZ8Foh6yYYIoSGub8HY9oRBSjMc9yM4mSBc0YyRUVGQ1DLgWNvUZjnkkSy5DGHULS23wtBTlLCBcrBgu9ReRlW71nZxlLFooqon9uSNWKwlQecho3G6Tye1RSSjOZkR/abSNqa3OH0DinWYuENFlwqRsiZDFFgxl0F1IpkJE1G47jNHiSSpUTMCSlKmNlH/YYyaSxVDIhEc1ZzhNG7Ley3yL4f8RAnWl+BAUNsyb7LaZxuGbJbbkqZx/yRmPqjwbDYY90ycWLH//x/FmjP5hf9i79KYy8dfqXozdOizivChrRmP5KFcXuSIqcKcVusXNdsEWsh4c80X9fypXMN39gc0pFzulKYvs/BVObTzlVujfafFLgHmz2iyyUzk1jfDV6ObGae8KILHimZ1O1+axH/i2zzSctWi7t0CVnRsPPNGaK22lqxbRwmnBjXk9EoD8zYkO7dvP75gtfofbetT8cDi6vjP5XVJVTXjMlNn+IiH3USzdfU9MCm3LEqe5cw3ZMa7z5c/PFNCc02Xy2wzM4ZaaV/xRL27pmmy8UVPvTWQ8UB9PBq/nAH8+0D+6clK4AF06HOEy849hs7Zp6NFdMT7lvNBoRWxKLZpeJFReABxGEMQRVzrLu03a7jQMpiziArfu8rQdgBY9j2rU4aHY06AGI/WooQNylSkZFyBcx88hLRCzEGPmtYGRdLDgEqYfgxbVKrMB6g1dvov+4VouJqNIk9POdIySYzGA7S+cOlnrhWvKQuRYLzXtSHd2eUfMefRHyAgIM1lamlPht3tfCF39LiNMAtoTGrZi7c07zRs/9jrwGbpGAICSkizaJAAFZh4QykUg4hbCMgyyEYZxxIiAwwIkkkRFf8nDzlThrGjKYC0MJSJfGK3QtwWXdbax6Qr53yxj1ijw0vrGng64xNh3YbGfYg9L+BHMiCUuMjrMdC7gRvc266JxCcJCUuOct2Fdzx2zAUgUw6kcjwEr6viJhLQtVFwGg+fFpRQRf1qT864gZRsh5xeaTuvEURCHCulOrygy2vQhgycLcExSIsNslTnILHAdYQAmXvZk/G4x8koHnwMOYSgqqIpqAb+ezfs0Oo7l11B4PgA9Mzdz8IxdL2R3DaTVrU4/O2Mq3p+XRNIXc5t45FnIBL2GLgcJFjidTAWTrALzlz8Hco+O/Avq3JVe09riiJIgHOzYfQefmIcW5zCk6WMkCtlKFxMVFizxFQgHXPIV/TgJOAG7fs3H0Zqx5s/ZnPr18WK05iIAJEFcelVM5APOlMnBv3P+e5+sSLAsGf9wmoRkJK4HjsQ8sLHI4O8iFrnPpD/2ZT36aXI3Kg3MqQH9oenlwTj0wTmE1lVm+giSAgK1jcU9HbzjzJ2TWezn0d7xpBvtXw/loTHhEJpg/JjPyy2D2mpxXTXhMZEk33y6RxRn7KybPZ1fBYNyf+CPIbxBef83MB6XsSxiMpz7sYTCeXe0McU2+AfbWuaNJfu4N55Bv3U75oWO/OADkbTx+k5LSTHcX5C1iwhXqMYyeFjGR0SJbVNfIBPpN4pyKBWdra1VBp9TQsSo6pY5ORUlnTwvurkwlJm2zvFCCbBkqg6DaERKEXBkRnV0RUZYdWPpWag6ZQj1gqtvuebXI0Ny4rTB626oeyqiUh0DSz9rkb2W5/Ow5tt8hN2E6Lnd5jlP2KmicWVbRnhbfx7Rs7dh8FbCaZAWBGiaF5L69MMCXVi270zQGO8wSTPL/Q1GDJbHUZRvaD15qI01p822nYrQdsTZj7/6As0CiANI4YK2EfoBzxwy1B8Up8FF/BlHcG/rTvu+Oem9cHsE5t5v7nOZlcD2COnGH691RPyR5Pp65f28eUF5d3EHJUsFDpQSITJ4tcyA4dGeNXPwKVdK76nfMkbB1zJPWA2fEppzacLNJsAKIq7dGKCXAmyEmPX1zLANVsdDsYav48aSBP2B2bf0/Sdt71u4cxOv/hyQq3BS2yBOT4Z9gnGOMYwSfTJoE4/jhAgPy+Tfm8hNp3CTu++bh7nUovDVxcEO+B5quzWFx1X8vnh/6z3DVaSAad9VATX557U98ogOjw5Hd7hwOlpdIuj+C+cpxWo0A2sO72KF9x854fo1F59acqb81Ewxix8xDXjjU9taYcgMertt/3IrS14ZoHvX2xRFviyOOfmg3T7b14pP6tkhvfFmZZgF79CyUfB8C9PKT4KmSpd6V2NvVfhlyaidHirpvwMqj9pXUXbHNZlAzAzIj+D0IsOQLAl30BUFCuQgCW/LZZ5hbmsSGNvXTjsyhfi+/xZJGgRlq7OZgZmAfwDHlNEzzKoBqsmAJRfaAO2wQsSxiAUxd8pVZnOIV0T4reT21KvB6eo095cLsUPE0B6nd+rvcjkWrj3SOva56NIoCamW5ztmZUQj8AHUBLeK865gRD/d5cpV9t3B0MaRtwHs4C3JV4CPOmsVpV18E4OCJ3Si5RXNqL3Yn5eu3ORCU36asC2S4M69dShebPxOmJO6ykq8I3EO1N5SVjXf41NM+RA2ZpZPKSbmV6wdIEi71jAuA5kUIN1qx6jpFvjx74ejcstzh2CQe0HB4hC76z8vokgWoy13awhe5y7Puqz8JbBFhK7PKpVRhPqi/8HSc1t7rUVWB9l9nX8CugLPvoBSl1IpBuxRk/RdQSwMEFAAAAAgAZJRBXQCsYQlZBAAAHAoAABUAAABpbmdlc3RhL2JkL2Rlc3Rpbm8ucHmlVttu4zYQffdXTNUXqTCEPhswCl+UrlFfsrbc7jYIDEYcOdyVSJWkdtcJ8jF97MM+9RPyYx1KtB0j7ma3DQKY9Fx4eM7M0EEQjNFYIRUwyYrHz1ZkCsJxnb0fD6MeZExvmYFMlVWBlpku1JVBbUFwLCtlUVqEHbCqEBnLxOPfEjjCjdKacWXiIAg6gvwogFNKfrPfVUxyykv/Fe90RovpejYfrKAPV4HgQReCrBAu96bdEUTK51ZWWVa4Rakkcta4aiTjBqXbsMzWrBB3/pvrzng8pbQOyGiZDNIE0sFwmsDkAuaLFJI3k1W6gnv7AGEH6E9wGE5+nsxTuFxOZoPlW/gleduFIxxv7kKLCX4dLEevBssuNMhgvFhT+i608A7WJvcBKKSTWbJKB7PL9PcunEI+tW3wk9VMnLOUaNXx/E7krtjpcMyB3WihQ11bFvWagzXaWksvQZwpKTGzobHeKfJxUumyAaJDnvtQypqKShm4VaXaPv4lkdZGSGiFZJo2gAUoLbYEMa8RLpWxW42r11NQMNvRZ1MHLhvPSQueX+0FvyYw1S6MvPHKy3vdeu13MTN2V2GYF4rZ1jdXmoqyAAISfrEA/DV8fgpxuSseW7XhzKIVJYbeQLVts36qa4zI+/sWOpDsGrcMxlQ7ToDm8ndKsh4UDAyapuipGB7/JF9Yp6MTynPP7abSWDFN1BL9VCzspmBdMlOlNDp6nGSM8RNmtcWQKjfOnSQ2tP0mIIqOLJ6KdSDwacEE/q4pXZKKtaxiqT6G9q4fEMrgSUwLoXFvl2evoJFavmB3L92BxB65sXGYGr12IGhA0stND1pyx6tyw6KWDAiyNCxrB0goGRcIH9AR3BwCH1j2+JlFhzoqaPIQ2BdJfcZpMEyoe4PnhjwYJ9OExsPFcjGjeeBSPZz1m8xXyTIFmgGLvR+sKHaUwg9ttEN3JjQYLWazSeotnt0CZej89z24J2gjJPVTrV6keqrgD2q6+QJu2a5Z3rIMdQ88TlejGc1Zqs2G6YZPV8NFwYjiSouSacGadHRYqZznUGxf16h31MBL5OZW5Lbb5KZ3wqWXkNWkrdBtFhPFMGLcKa61oE+ShVqLFrxuXoaDcC/Ud0yBBeUPg6fj17V14GvfK3+m+p3l2+vfR53vgP8h/L/I276eL4rqj4rjGBZzGC3mF9MJHTNewPrSDaJewzTqQjWiUh8JvKOOydBQS71jbiKXwpCYGk1duIfqIIHI6ciYWtnujrPRw/3xm7or5lpVGy+xRROa+sag7bdP+HvEqh8UNLs9JYqEMe6Jz9rx7Ya3O4regKIupXHIMviOHmuKv25CmBFbSb8r6DF1ka4Q4ndKSFLjPnugb5I3o+l6nIxj2gbHtM1R/715T0gPBY+OzJN3CvdPcT18WfF29J1QKLjxUtONaXOU4avG/1eNLvjtVbJM3M+ZyRxCf8X1fE4/IMKfooiYvCqEsaHDcv3sAu7bzj9QSwMEFAAAAAgAbJRBXZ9c6+5DBgAAhQ8AABgAAABpbmdlc3RhL2JkL2V4dHJhY2Npb24ucHmtV9tu3DYQfd+vGKgwIKVrJWkf2m6xAVxngwTwJbA3KIogELgSZTORSIWkfIU/oJ/RD+hT3vrqH+shKe1KttMmQAwDlkTOzJmZMxdHUbS4sJrlubj9JKngpuBUMVoxwwnfpXFHSrJqRjnTJ4xyVTcVt4w4CZlrXnNpWUWN0lQznTPoIHbSMorPmeUa3z4kaRRFE1HjjqX3Rsn+2YqaT0qtaipw171Rd9K/T/2dgleWhccrJTuRhtnTSqx6idd4nfSKGyYLZgi/TRFum48Vq/JTXl/2ApZf2Ek4TFMgPxVnyvSH3ORarITOmFW1yFV/r1InmztncL0/WQPnxgqpJpPdw4PjN3vLnWz3cP/13mK5Q3OKjhd7i90liWJKeSUgzjP3DBlWKDioEMop1fCxgL+ILj5nXE6J5bZllbgK7/Ti6HCfGl6IQploY+rVwe7RYn9xsNzZg7X7CL6niH5/uTha3NX3jGYh9YdHzxdH9Nsfdy5MATmaTCYFLylDcnXBdKbZeVyUU9BFFkKeADe3yrlR8vyUZZ4uyWxC+EH+d1kBzgy49rHl7osk6HlMjkN562hmBPGqjyOBhrnSCHEDLuClgRQnzXMljdWt0J5bzkZg33xNnVSq87inTNraPEkhUbovcbT1cmt/63irjBIv21lzLkGDo1LceZXQY4rwOXJ/V4X/0wcez2U0cHZ+PXi5CcfXISg32bXHd5M2TMNzGzDfpVk8AOICW68QIls3sx4hnEJJubxoxD5Jrco6hTGuIU2y4BfzF6wyPEm6hHk4WV+3MZcnQiKYiCAYx1YVyzrlg1Q+mMI9zml5+HzHNQgvSJfIBIdedsUGSUvpWDhjOOYXIBKo3HCtHBA2owpaDqgUFaozd6Q441frJAop0G6QA59BSJUA3qJOdBwydS7sKQUXUnggeW7jxBV6HmD6bJZQ0BSpK58MhR+7Uo/vlUOCGASln6d01IctGsdkyJp0HQMdPxRUp3CjJ4ii4K0Byuso8COajUxFPjxZxQXaWORiJl26cWD4SSsd+WakEZgifiBQtN3FcUo/JDeBaL5XxWBwNqYDjHVA53dgP3rkQQa8mttWywC77wO6tSwLrSteB2ykpKNOJ3yvrDrhUCjraZFdj3TcpG5i9M0H5NHZ+ur/mAU+hPhLcAbulSSV9WIpvxDGmjjZkKpz4sDNn8H7utu4MSCMKl192thhxrBghYm9Pk9FT0Muc+UgzKPWlts/R0nyduN79G5ctIMJ+9V1i69KfVix/EMG0sx/fPJkU8lLzTgZVSnccY0Y5uoVuvJmAbj9p4JbbDzTXYuXGJKtdFULVNIfoaebijWc4t5ikk68qW3au6OCDENjcDag5/nO8vCY4ppdxON5k4DqspVerEKoK/WeAq769q+PrYBdh9qPE552phbV2mM6a3l1BoueMN5WcAgPtZCtVWZGDMING2453IRgKFkKlw8lEWhddBaIjrhXh2uuDNHRMPKdAHCF5Yi72FDbGI5VQBRoC8q6Kd9D3BO1sKzftahUKOWZh7fCkIPvhsrbvw0mgaGDQzfvEGaElfl+SZcMDCXPTUxCdFMH1bRYjnTFkrTP7pd10n5efllJ9RUSpITxhTBouZ43m+EbP/3lpydTeup/7ZWQJfrLcBZD5jtqND55L7RGp5uRa4WhX2GA3dcejG9vdsLYuEWgMPMh1QNWFx70Vu1C4OVTZnoE47Wgd62bKoUAQ3ObSojTHEtbfYkZEm3QDDWvn1HhGAE5VAdnXXiCkyD5Yvlqf4HFRhKMsl8dXww3ngVg34q993vQm+Xutxpxg0XQTblpwGnm15EPBYbHGvjNqPkVZQrS2suNpf+YjYPu9NB4HI7IUBKfH49DVRtp2fIzFhzForPmVlG+jcb9Ino3JaRyvtQtT1LXT8JqdLmW2Si9S2LCfy7B0LNwsPGdfALRgcL0cPUYokHPF8ev39z+eeyq2Lus/eLKqhNFJauQYIZSMy5aYPntpwvXSzuao581AgXMXb+yfGTt3kL4JdNrOlJx72e4RvoZjKckPdfAEEaSH1ZFWzcmvh7MolkIS7qZaskN2HR/gI3M923Fy3521xlTJxqWr1t3Bq+fX4bGXo+AexBD4Jusu87i8/4Nl6mxN1+9T/0LUEsDBBQAAAAIAHeUQV3OQbHiGgcAAM8PAAAaAAAAaW5nZXN0YS9iZC9jZGNfcG9zdGdyZXMucHmlV1tv3DYWfp9fwVULjJSMtc0WiwJqJ4VrT7sGHCfN2OjDZKJwpDNjthSpkNTEzuUH9WGf8tZX/7GeQ1Jz8QVFUcGwJJI6t+87l0mS5Oj4iC245bVmoBhIJvWK1cBeaOtWBqY/n7L0l8PTjFVa4Xqla7EUFa/EzWfF5M3nFb7kg8GF5Y229H0ru5VQ7I0D68rwgVq9GbG3HbC1AAVMqEp2IijcqimY0uySX/uDQlnHJTdM8RrFTxRrja67Kqi1wDrL2Zt2pTvXdu4NS1ExGi05M9DKjX2KO7HmGePMGb6++d3SoWNYwHvRNUwzKxqBWsB+OyDLDVomWoGGeU8aYRtdMAlgdgJT8WYhyFV0v1uhibSmle2kwxfJLXN8gTcMykt42wkrnLYFe8dlKWGNcsYkCU2U7Jp1Cj3puEGdFN8WDCrVJHLXj3yQJMlANK02Djf6JycaGPQvLVc16sa/th4sjW6YfSu5rC6huWb9B3DlBmEzz9EI22+gXcrpuNMv1gigUBrdmJSnJ2eTQ7TcQF7pphUSUpO8JkeBpd+/+A4sotbwp7PX+fxx9iqnNR8GXClwpfCndPv05Gw6eXn+8eLF8eH55OPx5HRyPom7CIPTT/NH2ZdJRjqPnp9ePDu7ozV99Q41zNLZ61dz0jUv0mH6fTF7PZx/HA6zR8OPr6aPMxQxGNSwZJUBbkortUtBITFhxOhlnCzsqkRmolKeZMWA4YVRnki/HRkwbLipMDXMEKMhA8wNMtcUu/mxQgBrTucxUZDCKNGTGKIs6cFdCtNATkCSqnfCXbJgUL4AvKUZYVcFQ+iCK2EdoPdVDldQdQ5Swi9Nphi0o3P2hP348vkz1q7KSBUntPKOWvbL/yYvJ155qXhDQgqbZCP2IbFJ4dc/ZbmtKMPSbKNRLDEFXVS8NYSuB2xA5RRfB2Wk9B1b0sKO2HCvFgyz26Z4VYGDaVLVlf+ylGiGTiJc9G/kwaz1eGtl+NSA64zaMT5iv9DG/B3wL1QAjC8wmbSimoiifc0iZFtuOJYMaFoDmLttB1RyJCju60MtbKU9TyyYNbEkZz/cfEYTJFUads2pwoXa9Td48HDoa6PbuwHfoJ79Q4LEIJZrtN+kTrSaAOhqHQMmluGVjccsUZ2UydboiMiZVrB3NseAG2fJ6zQZ9qH3bgZR4T57Uhw8medkN68ATw6RB3S+F0bWYJNgaSKUgxUY2l8IjJ+jJ9twKek5u2MSrqbBi11Ze3aprgEjqiRj2uxoqnVH5Q7Br7Coa0WKkI/yHiVLqflfqqHyje9Ne4+Ats7P++1dOXHbr0R8kJWWShz6Czz1/7eMHoYS3aLlospbqJGW2IxC+S2YqGchaPPiG0baMPTEsnkxBLUW+DrETpEP2cFTpHeFGag9pkRnrJA/TH46Ofv30fNnz07ON4xuEMS+Y+QNd9VlNKoPBKVp8zBVltiRUcSHSsviPvKxJaKCmyO2s0oIbVtGvhTYDKVMm1niu0oyzz7txu9D4psTUh1PhMc5oqnbsIJ3eiVLcIFun2KwaRYor0vu88lsigqW91Fo+2XsmvcVmv6INmIFapxEOHC94VdlnCvG//0Kry2Cp4A1BseNfuxoQdVYkbCihlJD9Wp6ctb3JkNn0xbgN8xmeg6mMi77fo6Vy2pJw5fTqgLrFfE1V+9537FyNhVsiQHE72imagQyY7RnhhWrDicgfnnz/4JEGViRkoMGlLYHneIHa3jPHkf1YRoTNZZO7cj4DVuEEpWgvKdsQIKaZVnpDk/0jWm3QKKPCip3p0QSQvbhTimtKgpaGbErUfePNcc23VfIvn35ikjRK6tLTrD5/nV2cXo6YoWixrXXFvevbfFE8iii0hZW7LdEyN1WRxbPwNMZfIXZz+RlThYGui9p33uZUQYBw/7EYENdqsC71Jp7LV+wIxyaeOVC+AtK2o5mTGJdAPLmM/uaraECuxn+HfVI4keYArHhdezmD4nw6FghvPAurGCehsTaeBF924ITTs7QWp9P81ki6mSONrOQ712L7dKFYPRntuLC1zmWgQ7BCN6HBGX/wrYTBsgkOOzbfR3iKrwIMdqTIhw0t4SMbwvBvWjR1oWYOXnYSO/LdizYxwjXjwa7aRoFYAbSNBO7Vi8kDCX3Cukd2JRKD/nWjr8eFnwnfXBguD0HlLzGvMcGSyQ/OpyepwXmCjuc+pSwKrs1qaE/uBpLop1Ri8b3OL+hQw6IERiKKfZPhGvWR/k2O7C61u66hXGiF79iRidZQDjkPkKUO11Sw4kJg7zzDPmQNOB0rdEEH1kyiJLFlvjzKqGfaioNaYI7UVdcjm+0gbwuNZYazAtN3gTLb2V2EkGMn+9A2qMUdzag0dwBq06FLYOe1Ok9VY0dxKI3Yv+JXWl39A0Fs6bBN/JifIsljx75cOwNBH5l8CdQSwMEFAAAAAgAxJRBXWrJTU36AwAAIgkAABkAAABpbmdlc3RhL2JkL3JlY29uY2lsaWFyLnB5lVZLb9w2EL7vr2B5ktK1EPRUGBCKIo1P6QNpbosFMSuOdolQpEJSfiDIT+qpt14K1H+sM6RkrWsbbQXDMOf5zdtSyvfYedcZa6Az97+7S/HXn2iFxpiM80KbDoX1YjBx8OLThIKYPpgjuu/EOxD3fzjTgeh9GECcvCM1IGUR4YCB9DCKzg8jBAiNlHJjhtGHJEZwGqKgn1Fv+uAHET9ZsN0JhzsxyyS8TZvCbBrrj3Fh4DW65DebjcZehAf4oUJ3NA635NFtRYKDBTXHUV9uBH03Jp1EkWpIyGGXqppRdIXPXwlOtISsCQhaEbKKsVTy17fv3r75IIzeCo5Te/LiE1hx9f7nH8WI2mgfZU0A6s1qaiezkNyTzX9QGojpbsSqtx5S0WHAJEjoGrzFbkrE/RfHnx+F+kXWje6rYq0LE9Vv8dsMGI5YseBWeNdKo+WWqnbTSk+OAj3i1PfmFmNbSVWUiCgX2xybcdQTkHxoP4QJixsYDj4yana3K7+lyt447FbIg08nuc/CAeNkOQhS+PyQdrm4uHxcue0q0RsLcUF1KSy6qjzqJ0KrMZbi1yMZIP9OoTuTMy5V1XPILfZJeWfvKK9xGqr63FL0h/CfLRHW08um9DRaSix10FNz/NpxtfbNLJZQV88YKd2htGHt9IAm12e3cOcM7sVXrXjMWbzun8NnegxIowaqdO+lCH5yega3dHTWExdPOj3Tt+Kb2eSXx62wk90EOkAeErC2Wjkfc/Je84oRHykcUT1Tv+1zpVjB///vhXJsn6a4LiNQllIlw9k69Tw8r149xFLPQacpuDV2WmSmF0o5GFCp3ClKDWCcUrKspXnvQTjSIo24OSfewWDzOy9K7RO664VHW0WrQtqsMrz58JbALWKDp2nmEDUq4vXmeCa8HIIFwyGYUIyNXKoZUvN9OE4DZeAXfgXuiS6YMRneMm/KARAWxAEiikR1itBxfmiFkUci8Fm5gYAnP0WUJVFjA1ormC1X8uKioKOk0uYHyl4rC6XhLLyolfdJJC1H1NjKr88M7OS8thWfKYsJuMYLzbguIJuhFj4jd7qjESmbjw9Fk3PAPuO8d88yP1Py6fEjbSNoCmja5a7z2rhjK6fUX3wr8ynq11PU9Ucyz7E1EXpUbLXq535bbx3JPC1hRco0b7le/PdOrumdsVMTREKYkFf357B7vc9DFnjIzu/Pcn44j5jbtNwd4/LZ5yKryKcbmixCJ7DpMXUnHuS6THpey+Rm9/K9rrP3xN6LIVKgwciEFWq5IWPgvUYX+gdIcBUIUpU91E3yKibiHis6VHjbXoGNWLOhAgHpKeRPXpzgTsxe+N+aM1x0WzVc3/8GDbXU31BLAQIUAxQAAAAIAESUQV2bVl1EXgAAAGUAAAATAAAAAAAAAAAAAACkgQAAAABpbmdlc3RhL19faW5pdF9fLnB5UEsBAhQDFAAAAAgARJRBXceuKJ5TAwAA+gYAAA8AAAAAAAAAAAAAAKSBjwAAAGluZ2VzdGEvbG9ncy5weVBLAQIUAxQAAAAIAESUQV2xIb/R8wkAAEYaAAATAAAAAAAAAAAAAACkgQ8EAABpbmdlc3RhL2FyY2hpdm9zLnB5UEsBAhQDFAAAAAgAxJRBXenfvIhiAAAAagAAABYAAAAAAAAAAAAAAKSBMw4AAGluZ2VzdGEvYmQvX19pbml0X18ucHlQSwECFAMUAAAACABLlEFdgRM6bVACAAA6BQAAHQAAAAAAAAAAAAAApIHJDgAAaW5nZXN0YS9iZC9zcWwvcG9zdGdyZXNxbC5zcWxQSwECFAMUAAAACADElEFdJX8QxKsDAABtBwAAFgAAAAAAAAAAAAAApIFUEQAAaW5nZXN0YS9iZC9jb25leGlvbi5weVBLAQIUAxQAAAAIAPCUQV0th+2/LggAAPAVAAAXAAAAAAAAAAAAAACkgTMVAABpbmdlc3RhL2JkL3NpbXVsYWRvci5weVBLAQIUAxQAAAAIAGSUQV0ArGEJWQQAABwKAAAVAAAAAAAAAAAAAACkgZYdAABpbmdlc3RhL2JkL2Rlc3Rpbm8ucHlQSwECFAMUAAAACABslEFdn1zr7kMGAACFDwAAGAAAAAAAAAAAAAAApIEiIgAAaW5nZXN0YS9iZC9leHRyYWNjaW9uLnB5UEsBAhQDFAAAAAgAd5RBXc5BseIaBwAAzw8AABoAAAAAAAAAAAAAAKSBmygAAGluZ2VzdGEvYmQvY2RjX3Bvc3RncmVzLnB5UEsBAhQDFAAAAAgAxJRBXWrJTU36AwAAIgkAABkAAAAAAAAAAAAAAKSB7S8AAGluZ2VzdGEvYmQvcmVjb25jaWxpYXIucHlQSwUGAAAAAAsACwDvAgAAHjQAAAAA"))).extractall(".")
print("Paquete ingesta instalado")

In [ ]:
import inspect, os, time
import pandas as pd
import psycopg
from IPython.display import Code, display
from sqlalchemy import text

from ingesta.logs import configurar_logs
from ingesta.bd import simulador, extraccion, destino, cdc_postgres
from ingesta.bd.conexion import crear_motor, crear_esquema
from ingesta.bd.reconciliar import reconciliar

configurar_logs(nivel="WARNING")      # solo avisos y errores en pantalla; hoy miramos tablas
LANDING, FECHA = "landing", time.strftime("%Y-%m-%d")
os.environ["BD_PASSWORD"] = "tienda-colab"   # base desechable de Colab; en el kit va en .env
engine = crear_motor("postgresql", "127.0.0.1", 5432, "tienda", "tienda")
wh = destino.abrir("tienda.duckdb")

def ver(funcion):
    display(Code(inspect.getsource(funcion), language="python"))

def sql(consulta, **params):
    with engine.connect() as c:
        return pd.read_sql(text(consulta), c, params=params)

crear_esquema(engine)
print(simulador.sembrar(engine))      # 2 000 clientes y 50 000 pedidos con 90 días de historia
sql("SELECT estado, COUNT(*) AS pedidos, ROUND(SUM(total)) AS total FROM pedidos GROUP BY estado ORDER BY 2 DESC")

## Parte 1 · Carga completa

La forma más simple: leer toda la tabla y reemplazar el destino. Siempre es exacta... en el momento en que corre.

In [ ]:
ver(extraccion.carga_completa)
extraccion.carga_completa(engine, wh, "pedidos_completa", LANDING, FECHA)

Con 50 mil filas tarda menos de un segundo. Con 500 millones, y cada hora, el DBA de producción te va a buscar.

## Parte 2 · Carga incremental por marca de agua

Solo traemos lo que cambió desde la última corrida: `WHERE actualizado_en > :desde`.
Vamos a mantener **dos destinos** para comparar:

- `pedidos_inc0`: sin ventana de traslape (`lookback_seg=0`)
- `pedidos_inc`: con traslape de 5 minutos (`lookback_seg=300`)

In [ ]:
ver(extraccion.carga_incremental)

In [ ]:
def incrementales():
    filas = [extraccion.carga_incremental(engine, wh, t, LANDING, FECHA, lookback_seg=lb)
             for t, lb in (("pedidos_inc0", 0), ("pedidos_inc", 300))]
    return pd.DataFrame(filas, index=["sin traslape", "con traslape"])[["filas_leidas", "watermark", "segundos"]]

incrementales()     # primera corrida: no hay marca de agua, así que lee todo

Ahora la tienda trabaja: altas, cambios de estado, cancelaciones y algunos borrados.

In [ ]:
print("Actividad:", simulador.paso(engine, 300, semilla=987654))
incrementales()     # esta vez solo lee lo que cambió

Observa la columna `filas_leidas`: unas decenas o cientos contra 50 mil.
La ventana de traslape vuelve a leer algunas filas de los últimos minutos; como la carga es un **upsert**, releer no duplica nada.

## Parte 3 · Trampa 1: los borrados

In [ ]:
pd.DataFrame([reconciliar(engine, wh, t) for t in ("pedidos_inc0", "pedidos_inc")]).set_index("destino")

`sobran_en_destino`: pedidos que **ya no existen** en el origen siguen vivos en el warehouse.
Una consulta incremental nunca puede ver una fila que ya no está. Las salidas típicas:
borrado lógico (`eliminado_en`) en el origen, reconciliaciones periódicas de llaves o CDC.

## Parte 4 · Trampa 2: la transacción que confirma tarde

En PostgreSQL, `now()` es la hora en que **empezó** la transacción. Si una transacción larga actualiza un pedido,
otras transacciones más rápidas confirman después, y nuestra extracción corre en medio, la marca de agua
**rebasa** a ese cambio antes de que sea visible. Vamos a provocarlo a propósito:

In [ ]:
# Elegimos un pedido viejo: el simulador solo toca los 5 000 más recientes, así nadie choca con nuestro candado
victima = sql("SELECT id FROM pedidos WHERE estado = 'pagado' AND id < (SELECT MAX(id) - 5000 FROM pedidos) "
              "ORDER BY id DESC LIMIT 1")["id"][0]

lenta = psycopg.connect("host=127.0.0.1 dbname=tienda user=tienda password=tienda-colab")
lenta.execute("UPDATE pedidos SET estado = 'enviado' WHERE id = %s", (int(victima),))   # transacción abierta
print(f"1) Una transacción lenta cambió el pedido {victima} a 'enviado', pero aún no confirma")
time.sleep(1.5)

simulador.paso(engine, 50, semilla=11)
print("2) Mientras tanto, otras 50 operaciones rápidas confirmaron")
display(incrementales())
print("3) Corrió la extracción incremental")

lenta.commit(); lenta.close()
print("4) Por fin confirma la transacción lenta")
display(incrementales())
print("5) Corrió otra vez la extracción incremental")

In [ ]:
estado = lambda t: wh.execute(f"SELECT estado FROM {t} WHERE id = ?", [int(victima)]).fetchone()[0]
pd.DataFrame({"origen": [sql("SELECT estado FROM pedidos WHERE id = :i", i=int(victima))["estado"][0]],
              "sin traslape": [estado("pedidos_inc0")], "con traslape": [estado("pedidos_inc")]}, index=[f"pedido {victima}"])

Sin traslape, el cambio **se perdió para siempre**: su `actualizado_en` quedó por debajo de la marca de agua.
Con traslape de 5 minutos, la siguiente corrida lo volvió a leer. El traslape debe ser mayor que la transacción más larga
que esperas en el origen.

## Parte 5 · Cargar sin duplicar

Tomamos la **misma** extracción y la cargamos dos veces, como pasaría si un reintento repite el lote.

In [ ]:
lote = sql(extraccion.CONSULTA_COMPLETA)
for i in range(2):
    destino.insertar_ingenuo(wh, "pedidos_ingenuo", lote, "ingenuo")
    destino.upsert(wh, "pedidos_upsert", lote, "upsert")
pd.DataFrame({t: [wh.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], len(lote)] for t in ("pedidos_ingenuo", "pedidos_upsert")},
             index=["filas en destino", "filas en el lote"])

In [ ]:
ver(destino.upsert)

## Parte 6 · CDC: leer el log en lugar de las tablas

Toda base transaccional escribe cada cambio en un log antes de aplicarlo (en PostgreSQL, el **WAL**).
Con la decodificación lógica podemos leer ese log como un flujo de eventos `INSERT`, `UPDATE` y `DELETE`.

1. Creamos un **slot** de replicación: PostgreSQL guarda los cambios hasta que el slot los confirme.
2. Tomamos la foto inicial (después de crear el slot, para no perder nada).
3. A partir de ahí, solo leemos el log.

In [ ]:
cdc_postgres.crear_slot(engine)
extraccion.carga_completa(engine, wh, "pedidos_cdc", LANDING, FECHA)   # foto inicial
print("Actividad:", simulador.paso(engine, 300, semilla=21))

# Así se ve el log crudo (peek = mirar sin consumir)
sql("SELECT lsn::text, xid::text, data FROM pg_logical_slot_peek_changes('bsg_ingesta', NULL, 12)")

In [ ]:
ver(cdc_postgres.leer_y_aplicar)
cdc_postgres.leer_y_aplicar(engine, wh, "pedidos_cdc")

Detalles que hacen que esto sea confiable:

- **peek y luego avanzar:** si la carga falla a la mitad, los cambios siguen en el slot. Entrega *al menos una vez*.
- **compactación:** si un pedido cambió tres veces en el lote, solo aplicamos su último estado.
- **upsert y borrado idempotentes:** reaplicar un lote no rompe nada.

## Parte 7 · El veredicto: reconciliación

In [ ]:
print("Más actividad:", simulador.paso(engine, 200, semilla=31))
incrementales()
cdc_postgres.leer_y_aplicar(engine, wh, "pedidos_cdc")
extraccion.carga_completa(engine, wh, "pedidos_completa", LANDING, FECHA)

tablas = ["pedidos_completa", "pedidos_inc0", "pedidos_inc", "pedidos_cdc", "pedidos_ingenuo"]
veredicto = pd.DataFrame([reconciliar(engine, wh, t) for t in tablas]).set_index("destino")
veredicto[["filas_origen", "filas_destino", "faltan_en_destino", "sobran_en_destino",
           "duplicados_en_destino", "estado_distinto", "cuadra"]]

| Método | ¿Cuadra? | Costo por corrida | Cuándo usarlo |
|---|---|---|---|
| Completa | Sí (al correr) | Lee toda la tabla | Tablas chicas, catálogos, reconciliaciones |
| Incremental | No con borrados físicos | Lee solo lo cambiado | Tablas que solo crecen o con borrado lógico |
| CDC | Sí | Lee solo el log | Tablas grandes que cambian y borran |

## Retos

### Reto 1 · ¿Para qué sirve el índice?

In [ ]:
with engine.begin() as c:
    c.execute(text("ANALYZE pedidos"))   # sin estadísticas, el planificador decide a ciegas
consulta = "EXPLAIN ANALYZE " + extraccion.CONSULTA_INCREMENTAL.replace(":desde", "now() - interval '5 minutes'")
print("--- con índice ---")
print("\n".join(sql(consulta).iloc[:, 0][:4]))
with engine.begin() as c:
    c.execute(text("DROP INDEX idx_pedidos_actualizado_en"))
print("\n--- sin índice ---")
print("\n".join(sql(consulta).iloc[:, 0][:4]))
with engine.begin() as c:
    c.execute(text("CREATE INDEX idx_pedidos_actualizado_en ON pedidos (actualizado_en, id)"))

Con índice, PostgreSQL va directo a las filas recientes (**Index Scan**). Sin índice, cada extracción incremental hace un
**Seq Scan**: recorre toda la tabla de producción para encontrar unas cuantas filas. Observa también el `ANALYZE`:
después de una carga masiva, sin estadísticas actualizadas el planificador puede ignorar el índice.

### Reto 2 · Un slot abandonado
Simula actividad **sin** consumir el slot y observa cuánto WAL tiene pendiente nuestro consumidor (el *lag* del slot).

In [ ]:
retenido = lambda: sql("SELECT pg_size_pretty(pg_wal_lsn_diff(pg_current_wal_lsn(), confirmed_flush_lsn)) AS wal_pendiente "
                       "FROM pg_replication_slots WHERE slot_name = 'bsg_ingesta'")
display(retenido())
simulador.paso(engine, 500, semilla=41)
display(retenido())

Si nadie consume el slot, ese número crece hasta llenar el disco del servidor de **producción**.
Por eso los slots se monitorean, y se borran cuando un consumidor se da de baja:

In [ ]:
cdc_postgres.leer_y_aplicar(engine, wh, "pedidos_cdc")
display(retenido())
# cdc_postgres.borrar_slot(engine)   # descomenta cuando termines el laboratorio

### Reto 3 · Borrado lógico
Diseña (en papel o en SQL) cómo cambiaría la tabla `pedidos` y la consulta incremental si en lugar de `DELETE`
la aplicación hiciera `UPDATE pedidos SET eliminado_en = now()`. ¿Qué tendría que hacer el warehouse con esas filas?

## Cierre

| Hoy viste | Principio |
|---|---|
| La marca de agua sale de los datos, no del reloj | La fuente de verdad del avance es el origen |
| Traslape + upsert | Releer es barato si cargar es idempotente |
| Borrados invisibles para el incremental | Saber qué NO ve tu método |
| Slot de replicación y WAL | CDC: leer el log, no las tablas |
| Reconciliación | Si no lo comparas, no sabes si cuadra |

**Tarea:** corre la parte 1.3 del **kit local** (`kit_ingesta_1_3.zip`) con `docker compose`, primero con PostgreSQL
y luego con MySQL (`--profile mysql`), donde el CDC lee el **binlog**.

**Próxima sesión (1.4):** procesamiento en un solo nodo: pandas, Polars y DuckDB.